In [ ]:
%pip install pydeseq2

In [5]:
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

import pandas as pd

In [ ]:
from pathlib import Path
#define project root
project_root = Path.cwd().parent
data_dir = project_root / "data"
counts = pd.read_csv(data_dir / "GSE60450_merged_counts.txt", sep = "\t")
counts

In [ ]:
counts = counts.set_index("GeneID")
counts.dtypes


In [ ]:
counts_with_names = counts.copy()
counts = counts.drop(columns=["gene_name"])
counts = counts[counts.sum(axis=1) > 0]
counts = counts.T
counts

In [ ]:
# creat metadata
metadata = pd.DataFrame({
 "sample": [
        "GSM1480291", "GSM1480292",
        "GSM1480293", "GSM1480294",
        "GSM1480295", "GSM1480296",
        "GSM1480297", "GSM1480298",
        "GSM1480299", "GSM1480300",
        "GSM1480301", "GSM1480302",
    ],
    "cell_type": [
        "Luminal", "Luminal",
        "Luminal", "Luminal",
        "Luminal", "Luminal",
        "Basal", "Basal",
        "Basal", "Basal",
        "Basal", "Basal",
    ],
    "stage": [
        "virgin", "virgin",
        "18.5dP", "18.5dP",
        "2dL", "2dL",
        "virgin", "virgin",
        "18.5dP", "18.5dP",
        "2dL", "2dL",
    ],
    "replicate": [
        1, 2,
        1, 2,
        1, 2,
        1, 2,
        1, 2,
        1, 2,
    ]
})
metadata = metadata.set_index("sample")
metadata = metadata.loc[counts.index]
metadata["cell_type"] = pd.Categorical(metadata["cell_type"], categories=["Basal", "Luminal"])
metadata["stage"] = pd.Categorical(metadata["stage"], categories=["virgin", "18.5dP", "2dL"], ordered=True)
metadata

In [ ]:
print(counts.shape)
print(metadata.shape)
print(counts.index.equals(metadata.index))
print(metadata.columns)

In [ ]:
# create deseq2 dataset and run deseq2
dds = DeseqDataSet(counts=counts, metadata=metadata, design_factors=["cell_type", "stage"])
dds.deseq2()

In [ ]:
# run deseq2 stats and get results
stat_res = DeseqStats(dds, n_cpus=8, contrast=["cell_type", "Luminal", "Basal"])
stat_res.summary()
res = stat_res.results_df
res

In [ ]:
# add gene names back to results
gene_names = counts_with_names[["gene_name"]]
gene_names.index = gene_names.index.astype(str)
res_with_names = stat_res.results_df.copy()
res_with_names.index = res_with_names.index.astype(str)

# check index types of counts and stat_res results
print(gene_names.index[:5])
print(res_with_names.index[:5])
print(gene_names.index.dtype)
print(res_with_names.index.dtype)

res_with_names = res_with_names.join(gene_names, how="left")
res_with_names = res_with_names[
    ["gene_name", 'baseMean', 'log2FoldChange', 'lfcSE', 'stat', 'pvalue', 'padj']
]
res_with_names

In [21]:
res_with_names.to_csv(project_root / "quants" / "GSE60450_deseq2_results.csv", index=True)